# Feature Engineering

This notebook prepares the serve-level dataset for modeling by creating new variables from match context, serve characteristics, opponent profile, and tactical combinations.

The goal is to build features that are available before the serve so they can later be used for point-outcome prediction and serve recommendation.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/table_tennis_serves.csv")

df.head()

## Data Validation

Before engineering any features, we inspect the raw dataset to confirm it loaded correctly and that the outcome column has a sensible distribution.

In [ ]:
original_shape = df.shape
print("Dataset shape:", original_shape)
print()
print("Value counts for point_outcome:")
print(df["point_outcome"].value_counts())

In [ ]:
df["point_won"] = (df["point_outcome"] == "won").astype(int)

The target variable is `point_won`, where 1 means the server won the point and 0 means the server lost the point.

In [ ]:
leakage_features = [
    "return_type",
    "return_quality",
    "return_placement",
    "rally_length",
    "point_end_type",
    "rally_type_achieved",
    "chop_rally_outcome"
]
leakage_features

These variables occur after the serve, so they should not be used in the main prediction model. Including them would leak future information into the model.

In [ ]:
df["score_margin"] = df["server_score"] - df["receiver_score"]
df["total_points_played_in_game"] = df["server_score"] + df["receiver_score"]
df["is_tied"] = (df["server_score"] == df["receiver_score"]).astype(int)
df["is_trailing"] = (df["server_score"] < df["receiver_score"]).astype(int)
df["is_leading"] = (df["server_score"] > df["receiver_score"]).astype(int)
df["is_late_game"] = (df["total_points_played_in_game"] >= 16).astype(int)
df["is_deuce_or_later"] = ((df["server_score"] >= 10) & (df["receiver_score"] >= 10)).astype(int)
df["is_game_point_for_server"] = ((df["server_score"] >= 10) & (df["server_score"] > df["receiver_score"])).astype(int)
df["is_game_point_against_server"] = ((df["receiver_score"] >= 10) & (df["receiver_score"] > df["server_score"])).astype(int)

## Additional Interaction Features

Beyond basic game-state flags we add three interaction features that combine pressure context with serve or opponent information:

- **`spin_x_looper`**: The product of `spin_intensity` and `opponent_is_looper`. Looping opponents are more sensitive to heavy spin, so this captures whether a high-spin serve is being used specifically against a looper.
- **`score_margin_abs`**: The absolute value of `score_margin`. This is symmetric — it measures how far from a tie the score is regardless of who is ahead, which is useful for pressure modeling without encoding direction.
- **`is_high_pressure`**: A single boolean flag that fires whenever we are at deuce-or-later, game-point-for-server, or game-point-against-server. Collapses three correlated flags into one clean signal.

In [ ]:
# opponent_is_looper is needed for the interaction — create it here so we can reference it
df["opponent_is_looper"] = (df["opponent_style"] == "looper").astype(int)
df["opponent_is_chopper"] = (df["opponent_style"] == "chopper").astype(int)
df["opponent_is_attacker"] = (df["opponent_style"] == "attacker").astype(int)

# Interaction: spin intensity amplified against looping opponents
df["spin_x_looper"] = df["spin_intensity"] * df["opponent_is_looper"]

# Pressure distance: how far from a tie are we (direction-agnostic)
df["score_margin_abs"] = df["score_margin"].abs()

# Unified high-pressure flag
df["is_high_pressure"] = (
    (df["is_deuce_or_later"] == 1) |
    (df["is_game_point_for_server"] == 1) |
    (df["is_game_point_against_server"] == 1)
).astype(int)

print("Interaction features added.")
print(df[["spin_x_looper", "score_margin_abs", "is_high_pressure"]].describe())

## Ordinal Encoding for Opponent Skill Level

The `opponent_skill_level` column contains ordered categories. Converting it to a numeric scale lets models exploit this ordering directly.

In [ ]:
skill_map = {"beginner": 1, "intermediate": 2, "advanced": 3, "expert": 4}
df["opponent_skill_numeric"] = df["opponent_skill_level"].map(skill_map).fillna(2)

print("Value counts for opponent_skill_numeric:")
print(df["opponent_skill_numeric"].value_counts().sort_index())

In [ ]:
df["serve_spin_combo"] = df["serve_type"] + "_" + df["spin_type"]
df["serve_length_spin_combo"] = df["serve_length"] + "_" + df["spin_type"]
df["serve_placement_combo"] = df["serve_type"] + "_" + df["placement_zone"]
df["full_serve_combo"] = (
    df["serve_type"] + "_" +
    df["spin_type"] + "_" +
    df["serve_length"] + "_" +
    df["placement_zone"]
)

These features capture tactical serve patterns. A serve is not just defined by one attribute; its value often depends on the combination of type, spin, length, and placement.

In [ ]:
df["is_heavy_spin"] = (df["spin_intensity"] >= 3).astype(int)
df["is_low_spin"] = (df["spin_intensity"] <= 1).astype(int)
df["spin_length_interaction"] = df["spin_intensity"].astype(str) + "_" + df["serve_length"]

In [ ]:
combo_summary = (
    df.groupby("full_serve_combo")
    .agg(
        combo_attempts=("point_won", "count"),
        combo_win_rate=("point_won", "mean")
    )
    .reset_index()
)
df = df.merge(combo_summary, on="full_serve_combo", how="left")
df["combo_reliability"] = np.minimum(df["combo_attempts"] / 30, 1)

Serve combinations with very few attempts can have misleading win rates. The reliability score adjusts for sample size, giving more trust to serve patterns that appear more frequently in the dataset.

## Feature Validation

Before saving we run three sanity checks:
1. How many new columns were added vs the original dataset.
2. Whether any of the new numeric columns contain null values.
3. Which engineered features correlate most strongly with the target `point_won`, sorted by absolute correlation (top 10).

In [ ]:
# 1. Column count comparison
engineered_count = df.shape[1] - original_shape[1]
print(f"Original column count : {original_shape[1]}")
print(f"Current column count  : {df.shape[1]}")
print(f"Engineered features   : {engineered_count}")
print()

# 2. Null check on new columns only
new_columns = df.columns[original_shape[1]:].tolist()
null_counts = df[new_columns].isnull().sum()
print("Null values in engineered columns:")
print(null_counts[null_counts > 0] if null_counts.any() else "  None — all engineered columns are complete.")
print()

# 3. Top-10 numeric features by absolute correlation with point_won
numeric_new = df[new_columns].select_dtypes(include=[np.number]).columns.tolist()
correlations = (
    df[numeric_new + ["point_won"]]
    .corr()["point_won"]
    .drop("point_won", errors="ignore")
    .abs()
    .sort_values(ascending=False)
    .head(10)
)
print("Top-10 engineered features by |correlation| with point_won:")
print(correlations.to_string())

## Feature Summary

All engineered features grouped by category.

In [ ]:
feature_groups = {
    "Game State": [
        "score_margin", "total_points_played_in_game", "is_tied", "is_trailing",
        "is_leading", "is_late_game", "is_deuce_or_later",
        "is_game_point_for_server", "is_game_point_against_server"
    ],
    "Interaction": ["spin_x_looper", "score_margin_abs", "is_high_pressure"],
    "Serve Combinations": [
        "serve_spin_combo", "serve_length_spin_combo",
        "serve_placement_combo", "full_serve_combo"
    ],
    "Spin Flags": ["is_heavy_spin", "is_low_spin", "spin_length_interaction"],
    "Opponent": [
        "opponent_is_looper", "opponent_is_chopper",
        "opponent_is_attacker", "opponent_skill_numeric"
    ],
    "Combo Statistics": ["combo_attempts", "combo_win_rate", "combo_reliability"]
}

rows = []
for group, features in feature_groups.items():
    for feature in features:
        if feature not in df.columns:
            continue
        dtype = str(df[feature].dtype)
        if pd.api.types.is_numeric_dtype(df[feature]):
            win_rate_corr = round(df[feature].corr(df["point_won"]), 4)
        else:
            win_rate_corr = "N/A"
        rows.append({"group": group, "feature": feature, "dtype": dtype, "win_rate_corr": win_rate_corr})

summary_df = pd.DataFrame(rows, columns=["group", "feature", "dtype", "win_rate_corr"])
print(summary_df.groupby("group").apply(lambda x: x[["feature", "dtype", "win_rate_corr"]].to_string(index=False)).to_string())

In [ ]:
output_path = "../data/processed/table_tennis_serves_features.csv"
df.to_csv(output_path, index=False)

In [ ]:
import os

saved_df = pd.read_csv(output_path)
print("File saved successfully.")
print(f"  Path  : {os.path.abspath(output_path)}")
print(f"  Shape : {saved_df.shape[0]} rows x {saved_df.shape[1]} columns")

This processed dataset will be used in the modeling notebook.